##### Copyright 2024 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API Onboarding: Create a Children's Storybook

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/onboarding_storybuilder.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

<!-- Community Contributor Badge -->
<table>
  <tr>
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/Giom-V" target="_blank" title="View Giom's profile on GitHub">
        <img src="https://github.com/Giom-V.png?size=100" alt="Giom-V's GitHub avatar" width="100" height="100">
      </a>
    </td>
    <td bgcolor="#d7e6ff">
      <h2><font color='black'>This notebook was contributed by <a href="https://github.com/Giom-V" target="_blank"><font color='#217bfe'><strong>Giom</strong></font></a>.</font></h2>
      <h5><font color='black'>See <a href="https://github.com/Giom-V" target="_blank"><font color="#078efb"><strong>Giom's</strong></font></a> other notebooks <a href="https://github.com/search?q=repo%3Agoogle-gemini%2Fcookbook%20%22Giom%22&type=code" target="_blank"><font color="#078efb">here</font></a>.</h5></font><br>
      <font color='black'><small><em>Have a cool Gemini example? Feel free to <a href="https://github.com/google-gemini/cookbook/blob/main/CONTRIBUTING.md" target="_blank"><font color="#078efb">share it</font></a>!</em></small></font>
    </td>
  </tr>
</table>

## Overview

Welcome to the Gemini API! In this guide, we'll walk through a "fil rouge" project—a continuous single project to help you learn all the basics of the Gemini API.

Here, we will create a **Children's Storybook generator**. Step-by-step, you will learn:
1. **Basic Prompting**: Generating simple text with the Gemini API.
2. **Error Handling & Retries**: Making your code robust against transient errors.
3. **Structured Outputs**: Defining strict JSON schemas to design characters and plot outlines.
4. **Search Grounding**: Validating facts using Google Search.
5. **Multimodal Generation (Imagen & Veo)**: Generating images, videos, and audio to bring the story to life.

## Setup
### Install Dependencies
First, we need to install the official `google-genai` SDK and `pydantic` for structured generation.

In [ ]:
%pip install -U -q "google-genai>=1.0.0" pydantic tenacity

### Set up your API Key

To run this notebook, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. See the [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) quickstart if you're not sure how to retrieve it.

In [ ]:
import os
from google.colab import userdata
from google import genai
from google.genai import types

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

Select the model you want to use. We recommend a fast model like `gemini-2.5-flash`.

In [ ]:
MODEL_ID = "gemini-2.5-flash" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro"] {"allow-input":true, isTemplate: true}

## 1. Basic Prompting

Let's start simple by asking the model to create a short, one-paragraph story concept.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Write a short 3-sentence story about a brave squirrel exploring the big city."
)
print(response.text)

## 2. Managing Errors with Retries

When calling APIs over the network, transient errors (like rate limits or temporary network blips) can occasionally occur. It's best practice to implement a retry mechanism.
We can use the `tenacity` library to easily wrap our API calls with retry logic.

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential
from google.genai.errors import APIError

# Automatically retry up to 3 times, with exponential backoff
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def generate_with_retry(prompt: str) -> str:
    try:
        resp = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt
        )
        return resp.text
    except APIError as e:
        print(f"API Error encountered: {e}. Retrying...")
        raise e

story_concept = generate_with_retry("Write a brief pitch for a story about magic animals.")
print(story_concept)

## 3. Structured Outputs: Character Creation

Narratives need interesting characters! Asking the model for freeform text is great, but for an application, it's much easier to receive data in a strict format like JSON.

We can use Pydantic to define our schema and let the model know exactly how to respond.

In [ ]:
from pydantic import BaseModel, Field

class Character(BaseModel):
    name: str
    species: str
    background: str = Field(description="A brief history of the character.")
    persona: str = Field(description="The character's personality traits.")
    visual_description: str = Field(description="What the character looks like, for later image generation.")

response = client.models.generate_content(
    model=MODEL_ID,
    contents="Create an adorable main character for a children's story.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Character,
        temperature=0.7
    )
)

import json
character_data = json.loads(response.text)
print(json.dumps(character_data, indent=4))

## 4. Structured Output: Story Skeleton

Let's now plan our story. We can build another schema to represent a list of chapters outlining the whole tale.

In [ ]:
from typing import List

class ChapterOutline(BaseModel):
    chapter_number: int
    title: str
    plot_summary: str

class StorySkeleton(BaseModel):
    title: str
    chapters: List[ChapterOutline]

story_prompt = f"""
    Create a 3-chapter story outline about this character:
    {character_data['name']}, the {character_data['species']}.
    Persona: {character_data['persona']}
    
    Chapter 3 should involve a trip to Paris during the Olympic Games.
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=story_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=StorySkeleton
    )
)

skeleton = json.loads(response.text)
print(json.dumps(skeleton, indent=4))

## 5. Generating Content & Search Grounding

Now we want to write the actual narrative for each chapter.

Particularly, we want Chapter 3 (set during the Paris Olympics) to reference real events (like specific stadiums, mascots, or sports) so it feels authentic. We can use **Google Search Grounding** to allow the model to pull in real-world context.

*(Note: Search Grounding provides real-time information retrieval from Google Search and appends citations to the generated text)*.

In [ ]:
chapters = []

for chapter in skeleton['chapters']:
    print(f"Generating {chapter['title']}...")
    
    prompt = f"""
        Write Chapter {chapter['chapter_number']} based on this outline: {chapter['plot_summary']}
        Make it engaging for children and stay true to the character: {character_data['name']}.
    """
    
    tools = []
    # Apply grounding specifically to Chapter 3 to fetch real Olympics facts
    if chapter['chapter_number'] == 3:
        tools = [{"google_search": {}}]
        prompt += " Incorporate real facts about the Paris Olympics (like locations or new sports)."

    resp = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=tools,
            temperature=0.4
        )
    )
    
    chapters.append({
        'num': chapter['chapter_number'],
        'title': chapter['title'],
        'text': resp.text,
        'metadata': resp.candidates[0].grounding_metadata if resp.candidates else None
    })

print("All chapters generated!\n")

Let's read the grounded Chapter 3 and inspect its Search Grounding metadata!

In [ ]:
ch3 = next(c for c in chapters if c['num'] == 3)
print(f"=== Chapter {ch3['num']}: {ch3['title']} ===\n")
print(ch3['text'])

if ch3['metadata']:
    print("\n[Search Grounding Information Used]:")
    for query in ch3['metadata'].web_search_queries:
        print(f"- {query}")

## 6. Bringing the story to life: Imagen 3

Stories are more engaging with pictures! You can generate images using the `imagen-3.0-generate-002` model. We'll take the character's visual description and create an illustration for the first chapter.

> **Note:** Image generation might have quota or tier restrictions depending on your API plan.

In [ ]:
from IPython.display import display

image_prompt = f"""
    A storybook illustration for children.
    Subject: {character_data['visual_description']}
    Style: watercolor, bright colors, whimsical.
"""

try:
    image_response = client.models.generate_images(
        model="imagen-3.0-generate-002",
        prompt=image_prompt,
        config=types.GenerateImagesConfig(
            number_of_images=1,
            output_mime_type="image/jpeg",
            aspect_ratio="4:3"
        )
    )
    
    # Display the generated image
    display(image_response.generated_images[0].image)
except Exception as e:
    print(f"Image generation failed: {e}")

## 7. Animation: Veo

We can also add motion to our storybook. Using the Veo video generation model, we can turn a text prompt or an initial Imagen output into a short video clip.

> **Note:** As of writing this guide, Veo integration requires specific allowlists on your Google Gen AI API or Vertex project. Therefore, we represent it generically below to illustrate the SDK capability.

In [ ]:
try:
    # Example showing how Veo video generation is structured:
    video_response = client.models.generate_videos(
        model="veo-2.0-generate-001",
        prompt=f"An animated cute children's book scene of {character_data['name']} the {character_data['species']} playing.",
        config=types.GenerateVideosConfig(
            duration_seconds=5,
            aspect_ratio="16:9"
        )
    )
    print("Video generation initiated successfully!")
except Exception as e:
    print(f"Video generation encountered an error or isn't enabled for this key: {e}")

## 8. Immersive Audio

Finally, let's add atmosphere! Using the audio output capabilities of the Gemini models (like `gemini-2.5-flash`), you can request sound effects or spoken dialogue. Here we are asking the model to narrate the story introduction directly.

In [ ]:
audio_prompt = f"""
    Please enthusiastically narrate this character introduction as if reading a children's book:
    Meet {character_data['name']} the {character_data['species']}!
    {character_data['background']}
"""

audio_response = client.models.generate_content(
    model=MODEL_ID,
    contents=audio_prompt,
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
    )
)

# Extract and play the audio data
from IPython.display import Audio

for part in audio_response.candidates[0].content.parts:
    if part.inline_data and part.inline_data.mime_type.startswith("audio/"):
        display(Audio(part.inline_data.data, autoplay=False))

## Next Steps

Congratulations! You've just built a complete pipeline for an AI-powered storybook.

### Continue your discovery of the Gemini API
* Explore the [Prompting Guide](https://ai.google.dev/docs/prompt_best_practices) to refine your character descriptions.
* Check out [Structured Outputs](https://ai.google.dev/docs/structured_output) for even more complex configurations.
* Dive deep into [Image Generation](https://ai.google.dev/docs/image_generation) to handle advanced parameters like style hints and negative prompts.